### 数据预处理（Data Preprocessing）

为了构建稳定且具有良好泛化能力的文本分类模型，我们对两个数据集进行统一的预处理，包括：

（1）AI Text Detection Pile  
（2）HC3 数据集  


#### 1. 数据加载与统一格式

两个数据集通过本地缓存的 parquet 文件加载，以提高运行效率。  
由于原始数据格式不同，我们将其统一转换为如下结构：

- `text`：输入文本  
- `label`：标签（0 = 人类文本，1 = AI生成文本）

统一格式后，将两个数据集合并为一个数据集，以提升模型在不同数据来源上的泛化能力。


#### 2. 标签标准化

为了保证一致性，我们对标签进行了统一定义：

- 人类生成文本 → 0  
- AI生成文本 → 1  

这一标准化过程确保了不同数据来源之间标签语义一致。


#### 3. 数据清洗

为了提高数据质量，我们进行了基础清洗操作：

- 去除缺失值（NaN）  
- 去除空文本  
- 去除文本首尾空格  
- 根据文本内容去重  

这些操作可以减少噪声数据，并避免训练过程中出现数据泄漏问题。


#### 4. 数据集合并

清洗后的两个数据集被合并为一个整体数据集。  
同时，我们保留了一个 `dataset` 字段，用于标识数据来源（例如 ai_dataset 或 hc3_dataset），以便后续分析模型在不同数据源上的表现。


#### 5. 数据集划分

合并后的数据集按照如下比例划分：

- 70% 训练集（train）  
- 15% 验证集（validation）  
- 15% 测试集（test）  

采用分层采样（stratified split），以保证各个子集中标签分布一致。


#### 6. 数据存储

为了提升效率和可复现性，处理后的数据集被保存为 parquet 格式：

- `train.parquet`
- `val.parquet`
- `test.parquet`

这种方式可以避免重复的数据预处理过程，并显著加快后续实验速度。

In [6]:
import os
import pandas as pd
from datasets import load_dataset

parquet_path = "datasets/ai_dataset.parquet"

if os.path.exists(parquet_path):
    print("Loading dataset from local parquet file...")
    df1 = pd.read_parquet(parquet_path)
else:
    print("Downloading and processing dataset...")

    ds1 = load_dataset("artem9k/ai-text-detection-pile", split="train")
    df1 = ds1.to_pandas()

    # human=0, ai=1
    df1["label"] = df1["source"].apply(lambda x: 1 if x == "ai" else 0)
    df1 = df1[["text", "label"]]

    df1.to_parquet(parquet_path, index=False)
    print("Saved to local parquet file.")

print(df1.head())
print("Shape:", df1.shape)

Saved to local parquet file.
                                                text  label
0  12 Years a Slave: An Analysis of the Film Essa...      0
1  20+ Social Media Post Ideas to Radically Simpl...      0
2  2022 Russian Invasion of Ukraine in Global Med...      0
3  533 U.S. 27 (2001) Kyllo v. United States: The...      0
4  A Charles Schwab Corporation Case Essay\n\nCha...      0
Shape: (1392522, 2)


In [3]:
import os
import json
import pandas as pd

file_path = "datasets/hc3_dataset.parquet"

if os.path.exists(file_path):
    print("Loading HC3 from local parquet file...")
    df2 = pd.read_parquet(file_path)

else:
    print("Processing HC3 dataset...")

    data = []

    with open("datasets/all.jsonl", "r") as f:
        for line in f:
            item = json.loads(line)

            # human = 0
            for h in item["human_answers"]:
                data.append({"text": h, "label": 0})

            # ai = 1
            for ai in item["chatgpt_answers"]:
                data.append({"text": ai, "label": 1})

    df2 = pd.DataFrame(data)

    df2.to_parquet(file_path, index=False)
    print("HC3 saved to local parquet file.")

print(df2.head(3))
print("Shape:", df2.shape)

Loading HC3 from local parquet file...
                                                text  label
0  Basically there are many categories of " Best ...      0
1  If you 're hearing about it , it 's because it...      0
2  One reason is lots of catagories . However , h...      0
Shape: (85449, 2)


In [7]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================
# 1. 路径设置
# =========================
ai_path = "datasets/ai_dataset.parquet"
hc3_path = "datasets/hc3_dataset.parquet"

# 检查文件是否存在
if not os.path.exists(ai_path):
    raise FileNotFoundError(f"File not found: {ai_path}")

if not os.path.exists(hc3_path):
    raise FileNotFoundError(f"File not found: {hc3_path}")

# =========================
# 2. 读取两个 parquet 数据集
# =========================
print("Loading parquet files...")
df1 = pd.read_parquet(ai_path)
df2 = pd.read_parquet(hc3_path)

print("Dataset 1 shape:", df1.shape)
print("Dataset 2 shape:", df2.shape)

# =========================
# 3. 只保留需要的列
# =========================
df1 = df1[["text", "label"]].copy()
df2 = df2[["text", "label"]].copy()

# 可选：保留来源信息，方便后续分析
df1["dataset"] = "ai_dataset"
df2["dataset"] = "hc3_dataset"

# =========================
# 4. 合并数据集
# =========================
df_all = pd.concat([df1, df2], ignore_index=True)

print("\nMerged shape before cleaning:", df_all.shape)

# =========================
# 5. 基础清洗
# =========================
# 去掉缺失值
df_all = df_all.dropna(subset=["text", "label"])

# 文本转字符串并去首尾空格
df_all["text"] = df_all["text"].astype(str).str.strip()

# 去掉空文本
df_all = df_all[df_all["text"] != ""]

# label 统一转 int
df_all["label"] = df_all["label"].astype(int)

# 去重（按 text 去重）
df_all = df_all.drop_duplicates(subset="text").reset_index(drop=True)

print("Merged shape after cleaning:", df_all.shape)

# =========================
# 6. 查看标签和来源分布
# =========================
print("\nOverall label distribution:")
print(df_all["label"].value_counts())

print("\nDataset source distribution:")
print(df_all["dataset"].value_counts())

# =========================
# 7. train / val / test split
# 比例：70 / 15 / 15
# =========================
train_df, temp_df = train_test_split(
    df_all,
    test_size=0.30,
    random_state=42,
    stratify=df_all["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("\nTrain shape:", train_df.shape)
print(train_df["label"].value_counts())

print("\nValidation shape:", val_df.shape)
print(val_df["label"].value_counts())

print("\nTest shape:", test_df.shape)
print(test_df["label"].value_counts())


os.makedirs("datasets/splits", exist_ok=True)

train_df.to_parquet("datasets/splits/train.parquet", index=False)
val_df.to_parquet("datasets/splits/val.parquet", index=False)
test_df.to_parquet("datasets/splits/test.parquet", index=False)

print("\nSaved files:")
print("datasets/splits/train.parquet")
print("datasets/splits/val.parquet")
print("datasets/splits/test.parquet")

Loading parquet files...
Dataset 1 shape: (1392522, 2)
Dataset 2 shape: (85449, 2)

Merged shape before cleaning: (1477971, 3)
Merged shape after cleaning: (1407894, 3)

Overall label distribution:
label
0    1035679
1     372215
Name: count, dtype: int64

Dataset source distribution:
dataset
ai_dataset     1385793
hc3_dataset      22101
Name: count, dtype: int64

Train shape: (985525, 3)
label
0    724975
1    260550
Name: count, dtype: int64

Validation shape: (211184, 3)
label
0    155352
1     55832
Name: count, dtype: int64

Test shape: (211185, 3)
label
0    155352
1     55833
Name: count, dtype: int64

Saved files:
datasets/splits/train.parquet
datasets/splits/val.parquet
datasets/splits/test.parquet
